In [1]:
import pdfplumber, re, json, os

In [2]:
pdf = pdfplumber.open("Road-User-Handbook-English.pdf")
print("pages:", len(pdf.pages))

pages: 212


In [3]:
def clean_text(text):
    """Strip page furniture and normalise whitespace."""
    if not text:
        return ""
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        if re.fullmatch(r"\d{1,3}", s):                  # bare page numbers
            continue
        if re.search(r"road user handbook", s, re.I):    # repeated footer
            continue
        lines.append(s)
    return " ".join(lines)

print(clean_text(pdf.pages[20].extract_text())[:400])

Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions are in place to help keep you safe as you develop your driving skills. They also apply when you drive in another state or territory. If you do not follow these restrictions, you’ll get a fine. You can a


In [4]:
pages = []
for i, page in enumerate(pdf.pages, start=1):
    cleaned = clean_text(page.extract_text())
    if len(cleaned.split()) < 30:      # drop image-only pages
        continue
    pages.append({"page": i, "text": cleaned})

print(f"kept {len(pages)} of {len(pdf.pages)} pages")

kept 208 of 212 pages


In [5]:
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/handbook_pages.jsonl", "w") as f:
    for p in pages:
        f.write(json.dumps(p) + "\n")
print("written")

written


In [6]:
def chunk_pages(pages, size_words=150, overlap_words=30):
    chunks, cid = [], 0
    for p in pages:
        words = p["text"].split()
        start = 0
        while start < len(words):
            window = words[start:start + size_words]
            chunks.append({
                "chunk_id": f"c{cid:04d}",
                "page": p["page"],
                "text": " ".join(window),
            })
            cid += 1
            if start + size_words >= len(words):
                break
            start += (size_words - overlap_words)
    return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks")

348 chunks


In [7]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/chroma")
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

try:
    client.delete_collection("handbook")
except Exception:
    pass

col = client.create_collection(name="handbook", embedding_function=ef)
col.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)
print("indexed:", col.count())

c:\Users\chris\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3146.74it/s]


indexed: 348


In [8]:
res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=5)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"[p.{meta['page']}] {doc[:200]}\n")

[p.31] Licences Alcohol and drugs When supervising a learning driver, a supervisor must: • have a blood alcohol concentration (BAC) under 0.05 • not have illegal drugs present in their system or be under the

[p.41] Safe driving behaviour Drive to road conditions Even if you’re driving at or below the speed limit, you may be driving too fast for road conditions such as curves, rain, heavy traffic or night-time. A

[p.199] driving over the alcohol limit (under 0.08, first offence) • not paying a fine • being medically unfit to drive. For some offences, police can suspend and take away your licence immediately. These inc

[p.201] Penalties Drink-driving offences that can result in an interlock include: • driving with a blood alcohol concentration of 0.08 or over • any repeat drink-driving offence • driving under the influence 

[p.21] Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisiona

In [9]:
# Failure case F1: the learner BAC chunk exists but ranks poorly
hits = [c for c in chunks if "must be zero" in c["text"]]
print(f"chunks containing the rule: {len(hits)}")
for h in hits:
    print(f"[{h['chunk_id']}] p.{h['page']}: {h['text'][:200]}\n")

res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=10)
target = hits[0]["chunk_id"] if hits else None
for rank, (cid, meta) in enumerate(zip(res["ids"][0], res["metadatas"][0]), start=1):
    mark = "  <-- TARGET" if cid == target else ""
    print(f"{rank}. {cid} p.{meta['page']}{mark}")

chunks containing the rule: 1
[c0031] p.21: Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions ar

1. c0048 p.31
2. c0065 p.41
3. c0324 p.199
4. c0327 p.201
5. c0031 p.21  <-- TARGET
6. c0067 p.42
7. c0069 p.43
8. c0326 p.200
9. c0047 p.30
10. c0051 p.33


In [10]:
import json

with open("data/golden/golden_set.jsonl", "r", encoding="utf-8") as f:
    golden = [json.loads(line) for line in f]

print("Number of golden questions:", len(golden))
print(golden[0])

Number of golden questions: 40
{'id': 'k01', 'dkt_id': 'AD013', 'question': 'If you take medicine and then drink alcohol -', 'answer': 'It can have a particularly bad effect on your driving ability.', 'type': 'known', 'in_kb': True, 'category': 'Alcohol and Drugs'}


In [11]:
from collections import Counter

print(Counter(q["category"] for q in golden))

Counter({'General Knowledge': 8, 'Fatigue and Defensive Driving': 4, 'Intersections': 3, 'Traffic Lights / Lanes': 3, 'Pedestrians': 3, 'other_jurisdiction': 3, 'unrelated_licence': 3, 'Alcohol and Drugs': 2, 'ICAC': 2, 'Seat Belts / Restraints': 2, 'vehicle_maintenance': 2, 'Bicycle Safety': 1, 'Negligent Driving': 1, 'Speed Limits': 1, 'product_advice': 1, 'insurance': 1})


In [12]:
for q in golden:
    if q["category"] in [
        "other_jurisdiction",
        "unrelated_licence",
        "vehicle_maintenance",
        "product_advice",
        "insurance"
    ]:
        print(f'{q["id"]} [{q["category"]}]')
        print("Q:", q["question"])
        print("A:", q["answer"])
        print()

o01 [other_jurisdiction]
Q: What is the speed limit in a school zone in Victoria?
A: None

o02 [unrelated_licence]
Q: How do I get a boat licence in NSW?
A: None

o03 [other_jurisdiction]
Q: What are the demerit point rules in Queensland?
A: None

o04 [vehicle_maintenance]
Q: How often should I change my engine oil?
A: None

o05 [other_jurisdiction]
Q: What is the fine for speeding in New Zealand?
A: None

o06 [unrelated_licence]
Q: How do I apply for a heavy vehicle licence?
A: None

o07 [product_advice]
Q: What tyre brand is best for wet roads?
A: None

o08 [insurance]
Q: How much does comprehensive car insurance cost in NSW?
A: None

o09 [unrelated_licence]
Q: What is the process for getting a pilot licence?
A: None

o10 [vehicle_maintenance]
Q: How do I replace the brake pads on my car?
A: None



In [13]:
q = golden[0]

print("Question:", q["question"])
print("Expected answer:", q["answer"])

res = col.query(
    query_texts=[q["question"]],
    n_results=5
)

print("\nRetrieved chunks:")
for rank, (doc, meta) in enumerate(
    zip(res["documents"][0], res["metadatas"][0]), start=1
):
    print(f"\nRank {rank} | Page {meta['page']}")
    print(doc[:300])

Question: If you take medicine and then drink alcohol -
Expected answer: It can have a particularly bad effect on your driving ability.

Retrieved chunks:

Rank 1 | Page 44
Safe driving behaviour Alcohol increases crash risk The more alcohol you have in your blood, the higher the risk of being in a crash. On Thursday, Friday and Saturday nights around 50% of fatal crashes involve alcohol. Crashes involving drink driving are generally more serious. It’s safest not to dr

Rank 2 | Page 42
Safe driving behaviour It’s impossible for you to estimate your own BAC, even if you think you know how many drinks you’ve had. Your size and weight, how tired you are, and variation in alcohol servings can all affect your BAC. The only way to be sure you’re under the limit is to not drink alcohol a

Rank 3 | Page 44
of alcohol. Black coffee, a shower or a large meal will not work. The only thing that reduces your BAC is time. If you’re planning to drink, make sure you organise a way to get home without

In [14]:
target_answer = q["answer"].lower()

matches = []

for c in chunks:
    if target_answer in c["text"].lower():
        matches.append(c)

print("Matching chunks:", len(matches))

for c in matches:
    print(f"\nChunk: {c['chunk_id']} | Page: {c['page']}")
    print(c["text"])

Matching chunks: 0


In [15]:
search_terms = [
    "medicine alcohol",
    "medication alcohol",
    "drugs alcohol driving",
    "medicine driving"
]

for term in search_terms:
    print("\n" + "=" * 60)
    print("SEARCH:", term)

    matches = [
        c for c in chunks
        if term.lower() in c["text"].lower()
    ]

    print("Matches:", len(matches))

    for c in matches[:5]:
        print(f"\nChunk {c['chunk_id']} | Page {c['page']}")
        print(c["text"][:500])


SEARCH: medicine alcohol
Matches: 0

SEARCH: medication alcohol
Matches: 0

SEARCH: drugs alcohol driving
Matches: 0

SEARCH: medicine driving
Matches: 0


In [16]:
res = col.query(
    query_texts=["taking medication and drinking alcohol effect on driving"],
    n_results=10
)

for rank, (doc, meta) in enumerate(
    zip(res["documents"][0], res["metadatas"][0]), start=1
):
    print(f"\n{'='*70}")
    print(f"Rank {rank} | Page {meta['page']}")
    print(doc)


Rank 1 | Page 47
the right decisions. Other illegal drugs also affect your driving. Do not drive if you’ve taken any illegal drugs. Medicines can affect your ability to drive Medicines can affect your ability to drive safely. They can cause drowsiness, blurred vision, poor concentration, slower reaction times and changes in behaviour. Some medicines that can affect driving are: • pain killers • medicines for blood pressure, nausea, allergies, inflammations and fungal infections • tranquillisers, sedatives and sleeping pills • diet pills • cold and flu medicines.

Rank 2 | Page 48
Safe driving behaviour Read the labels on your medication to determine whether it could affect your driving. If there’s a warning label that tells you not to drive, follow that advice. Examples of warning labels on medication. ©Pharmaceutical Society of Australia. Reproduced with permission 2020. Some labels say a medicine may affect your ability to drive. If you’re not sure, get advice from your doctor or ph

In [17]:
# Ground-truth source for k01, identified from the handbook
k01_source = col.query(
    query_texts=["Do not mix drugs and alcohol medicines"],
    n_results=5
)

for rank, (doc, meta, cid) in enumerate(
    zip(
        k01_source["documents"][0],
        k01_source["metadatas"][0],
        k01_source["ids"][0]
    ),
    start=1
):
    print(f"Rank {rank}: {cid} | Page {meta['page']}")

Rank 1: c0079 | Page 48
Rank 2: c0073 | Page 45
Rank 3: c0078 | Page 47
Rank 4: c0075 | Page 46
Rank 5: c0077 | Page 47


In [18]:
q = golden[1]

print("Question:", q["question"])
print("Expected answer:", q["answer"])

Question: In order to provide the minimum safe passing distance for bicycle riders, are you permitted to cross single and double continuous lines?
Expected answer: no.


In [19]:
res = col.query(
    query_texts=[
        "minimum safe passing distance bicycle riders crossing continuous lines"
    ],
    n_results=5
)

for rank, (doc, meta, cid) in enumerate(
    zip(
        res["documents"][0],
        res["metadatas"][0],
        res["ids"][0]
    ),
    start=1
):
    print(f"\nRank {rank}: {cid} | Page {meta['page']}")
    print(doc[:500])


Rank 1: c0133 | Page 78
calm. Only pass when you’re sure it’s safe. You can cross single and double white lines to pass bicycles, but only if you have a clear view of approaching traffic and it’s safe to do so.

Rank 2: c0132 | Page 78
the right to ride on the road. This includes riding in the middle of a lane. Sometimes they may need the full lane because of rough road edges or gravel. Be prepared to slow down and give them room to ride away from the kerb. Bicycle riders can ride two abreast (side by side) as long as they’re within 1.5m of each other. Another bicycle rider can overtake them. Keep your distance when overtaking When overtaking bicycles, you must allow a distance of at least 1m between you and the rider when the

Rank 3: c0230 | Page 144
Road lanes, lines and markings Double dividing line You can cross double continuous lines to enter or leave the road by the shortest route. Double continuous dividing lines. You can cross double lines with a broken line close to you to:

In [20]:
for cid in ["c0213", "c0133"]:
    result = col.get(ids=[cid])

    print("\n" + "=" * 80)
    print("CHUNK:", cid)
    print("PAGE:", result["metadatas"][0]["page"])
    print(result["documents"][0])


CHUNK: c0213
PAGE: 133
Overtaking and merging Overtaking bicycles When overtaking bicycles, you must allow a distance between you and the rider of at least: • 1m when the speed limit is 60km/h or less • 1.5m when the speed limit is more than 60km/h. To overtake a bicycle, you should have a clear view of approaching traffic. Only overtake if it’s safe to do so. If necessary, when overtaking a bicycle you can: • drive to the right of the centre of the road You can cross double • cross or straddle dividing lines (broken or dividing lines when overtaking a bicycle. continuous, single or double) • drive on flat painted islands and median strips. Bicycles can overtake a vehicle on the left. Overtaking on bridges When there’s a ‘No overtaking or passing’ sign on a bridge, you: • must give way to vehicles approaching in the opposite direction • must not overtake

CHUNK: c0133
PAGE: 78
calm. Only pass when you’re sure it’s safe. You can cross single and double white lines to pass bicycles, but

In [21]:
q = next(q for q in golden if q["id"] == "k02")

print("QUESTION:")
print(q["question"])

print("\nOPTIONS:")
print(q.get("options"))

print("\nANSWER:")
print(q["answer"])

QUESTION:
In order to provide the minimum safe passing distance for bicycle riders, are you permitted to cross single and double continuous lines?

OPTIONS:
None

ANSWER:
no.


In [22]:
q = next(q for q in golden if q["id"] == "k03")

print("QUESTION:")
print(q["question"])

print("\nANSWER:")
print(q["answer"])

QUESTION:
When driving at sunset or dawn on a dark day, what should you do?

ANSWER:
Turn on your lights on low beam.


In [23]:
res = col.query(
    query_texts=["driving at sunset dawn dark day low beam headlights"],
    n_results=5
)

for rank, (doc, meta, cid) in enumerate(
    zip(
        res["documents"][0],
        res["metadatas"][0],
        res["ids"][0]
    ),
    start=1
):
    print(f"\nRank {rank}: {cid} | Page {meta['page']}")
    print(doc[:500])


Rank 1: c0297 | Page 185
Warnings and road hazards At night or when there’s not enough daylight to see a person wearing dark clothing at a distance of 100m, your vehicle must have clearly visible: • headlights • tail lights • number plates • clearance lights and side marker lights, if fitted to your vehicle. Headlights Using your vehicle’s day running lights improves the likelihood of other road users seeing you. If your vehicle is not fitted with day running lights, you can use your headlights on low beam. Your headlig

Rank 2: c0298 | Page 185
lights You can use your headlights on high beam if you need to see further ahead, even if there are street lights. You must turn your headlights from high beam to low beam: • when a vehicle coming towards you is within 200m • when driving 200m or less behind another vehicle. You can also flash your high beam lights before you overtake another vehicle. 200m Turn your headlights to low beam when driving behind another vehicle. 200m Turn your hea

In [24]:
# Find the top 5 ChromaDB candidates for every known (k...) question

known_questions = [q for q in golden if q["id"].startswith("k")]

candidates = {}

for q in known_questions:
    res = col.query(
        query_texts=[q["question"]],
        n_results=5
    )

    candidates[q["id"]] = []

    for rank, (cid, meta, doc) in enumerate(
        zip(
            res["ids"][0],
            res["metadatas"][0],
            res["documents"][0]
        ),
        start=1
    ):
        candidates[q["id"]].append({
            "rank": rank,
            "chunk_id": cid,
            "page": meta["page"],
            "text": doc
        })

print("Questions processed:", len(candidates))

Questions processed: 20


In [25]:
for q in golden:
    if not q["id"].startswith("k"):
        print(q["id"], "|", q["category"], "|", q["question"])

i01 | Alcohol and Drugs | Can I drive if I've had a drink while I'm on medication?
i02 | General Knowledge | Do I need my headlights on at dusk?
i03 | General Knowledge | What should I check before getting on the motorway?
i04 | General Knowledge | Can I just double park for a minute to grab something?
i05 | Fatigue and Defensive Driving | It's pouring rain and I can barely see, what should I do?
i06 | ICAC | What happens if I get caught cheating on the driving theory test?
i07 | Intersections | Do I have to let pedestrians go when I'm turning?
i08 | Pedestrians | There are kids playing near the road, how should I drive?
i09 | Seat Belts / Restraints | My mate's kid won't put his seatbelt on in my car, who gets fined?
i10 | Traffic Lights / Lanes | Light goes yellow and no one's around, do I go through?
o01 | other_jurisdiction | What is the speed limit in a school zone in Victoria?
o02 | unrelated_licence | How do I get a boat licence in NSW?
o03 | other_jurisdiction | What are the de

In [26]:
for qid, results in candidates.items():
    q = next(q for q in golden if q["id"] == qid)

    print("\n" + "=" * 80)
    print(f"{qid} | {q['category']}")
    print("Q:", q["question"])

    for r in results[:3]:
        print(f"  Rank {r['rank']}: {r['chunk_id']} | Page {r['page']}")


k01 | Alcohol and Drugs
Q: If you take medicine and then drink alcohol -
  Rank 1: c0071 | Page 44
  Rank 2: c0067 | Page 42
  Rank 3: c0072 | Page 44

k02 | Bicycle Safety
Q: In order to provide the minimum safe passing distance for bicycle riders, are you permitted to cross single and double continuous lines?
  Rank 1: c0133 | Page 78
  Rank 2: c0230 | Page 144
  Rank 3: c0228 | Page 142

k03 | General Knowledge
Q: When driving at sunset or dawn on a dark day, what should you do?
  Rank 1: c0296 | Page 184
  Rank 2: c0291 | Page 182
  Rank 3: c0297 | Page 185

k04 | General Knowledge
Q: Is it an offence to obstruct clear vision of your number plates?
  Rank 1: c0307 | Page 190
  Rank 2: c0328 | Page 201
  Rank 3: c0306 | Page 189

k05 | General Knowledge
Q: Before driving on a freeway, which of the following should you do?
  Rank 1: c0279 | Page 176
  Rank 2: c0052 | Page 33
  Rank 3: c0295 | Page 184

k06 | General Knowledge
Q: What must you do if you miss your exit on a freeway?
 

In [27]:
import json

with open("data/golden/retrieval_candidates.json", "w", encoding="utf-8") as f:
    json.dump(candidates, f, indent=2, ensure_ascii=False)

print("Saved:", len(candidates), "questions")
print("File: data/golden/retrieval_candidates.json")

Saved: 20 questions
File: data/golden/retrieval_candidates.json


In [28]:
# Create a readable review of the top 3 retrieved chunks

with open("data/golden/retrieval_review.txt", "w", encoding="utf-8") as f:

    for qid, results in candidates.items():
        q = next(q for q in golden if q["id"] == qid)

        f.write("=" * 80 + "\n")
        f.write(f"{qid} | {q['category']}\n")
        f.write(f"QUESTION: {q['question']}\n")
        f.write(f"EXPECTED ANSWER: {q['answer']}\n\n")

        for r in results[:3]:
            f.write(
                f"RANK {r['rank']} | "
                f"CHUNK {r['chunk_id']} | "
                f"PAGE {r['page']}\n"
            )
            f.write(r["text"] + "\n\n")

        f.write("\n")

print("Created: data/golden/retrieval_review.txt")

Created: data/golden/retrieval_review.txt


In [29]:
# Find candidate source chunks using both the question and expected answer

source_candidates = {}

for q in golden:
    if not q["id"].startswith("k"):
        continue

    query = q["question"] + " " + q["answer"]

    res = col.query(
        query_texts=[query],
        n_results=5
    )

    source_candidates[q["id"]] = []

    for rank, (cid, meta, doc) in enumerate(
        zip(
            res["ids"][0],
            res["metadatas"][0],
            res["documents"][0]
        ),
        start=1
    ):
        source_candidates[q["id"]].append({
            "rank": rank,
            "chunk_id": cid,
            "page": meta["page"],
            "text": doc
        })

print("Processed:", len(source_candidates), "known questions")

Processed: 20 known questions


In [30]:
for qid, results in source_candidates.items():
    q = next(q for q in golden if q["id"] == qid)

    print("\n" + "=" * 80)
    print(f"{qid} | {q['category']}")
    print("Q:", q["question"])
    print("Expected:", q["answer"])

    for r in results:
        print(
            f"  Rank {r['rank']}: "
            f"{r['chunk_id']} | Page {r['page']}"
        )


k01 | Alcohol and Drugs
Q: If you take medicine and then drink alcohol -
Expected: It can have a particularly bad effect on your driving ability.
  Rank 1: c0079 | Page 48
  Rank 2: c0078 | Page 47
  Rank 3: c0069 | Page 43
  Rank 4: c0077 | Page 47
  Rank 5: c0071 | Page 44

k02 | Bicycle Safety
Q: In order to provide the minimum safe passing distance for bicycle riders, are you permitted to cross single and double continuous lines?
Expected: no.
  Rank 1: c0133 | Page 78
  Rank 2: c0230 | Page 144
  Rank 3: c0228 | Page 142
  Rank 4: c0205 | Page 127
  Rank 5: c0207 | Page 128

k03 | General Knowledge
Q: When driving at sunset or dawn on a dark day, what should you do?
Expected: Turn on your lights on low beam.
  Rank 1: c0297 | Page 185
  Rank 2: c0298 | Page 185
  Rank 3: c0296 | Page 184
  Rank 4: c0300 | Page 186
  Rank 5: c0299 | Page 186

k04 | General Knowledge
Q: Is it an offence to obstruct clear vision of your number plates?
Expected: Yes, at any time.
  Rank 1: c0307 | Pa

In [31]:
# Build an automatic ground-truth candidate list
# using the expected answer + question.

ground_truth_candidates = {}

for q in golden:
    if not q["id"].startswith("k"):
        continue

    # Search using the question + expected answer
    query = f"{q['question']} {q['answer']}"

    res = col.query(
        query_texts=[query],
        n_results=10
    )

    ground_truth_candidates[q["id"]] = [
        {
            "chunk_id": cid,
            "page": meta["page"],
            "text": doc
        }
        for cid, meta, doc in zip(
            res["ids"][0],
            res["metadatas"][0],
            res["documents"][0]
        )
    ]

# Save candidates
with open(
    "data/golden/ground_truth_candidates.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(ground_truth_candidates, f, indent=2, ensure_ascii=False)

print("Created ground-truth candidates for",
      len(ground_truth_candidates),
      "known questions.")

Created ground-truth candidates for 20 known questions.


In [32]:
# Display the first candidate for every known question

for qid, results in ground_truth_candidates.items():
    q = next(q for q in golden if q["id"] == qid)

    print(
        qid,
        "|",
        results[0]["chunk_id"],
        "| page",
        results[0]["page"]
    )

k01 | c0079 | page 48
k02 | c0133 | page 78
k03 | c0297 | page 185
k04 | c0307 | page 190
k05 | c0291 | page 182
k06 | c0225 | page 141
k07 | c0253 | page 158
k08 | c0297 | page 185
k09 | c0292 | page 182
k10 | c0049 | page 31
k11 | c0325 | page 200
k12 | c0149 | page 86
k13 | c0170 | page 103
k14 | c0206 | page 128
k15 | c0285 | page 179
k16 | c0124 | page 74
k17 | c0191 | page 118
k18 | c0081 | page 49
k19 | c0221 | page 138
k20 | c0150 | page 87


In [33]:
import numpy as np

# ---------------------------------------------------------
# Ground-truth candidate chunk for each known question
# ---------------------------------------------------------

ground_truth = {
    qid: results[0]["chunk_id"]
    for qid, results in ground_truth_candidates.items()
}

# ---------------------------------------------------------
# Calculate Recall@K and Reciprocal Rank
# ---------------------------------------------------------

ks = [1, 3, 5, 10]

recall_counts = {k: 0 for k in ks}
reciprocal_ranks = []

for qid, results in candidates.items():

    correct_chunk = ground_truth[qid]

    retrieved_ids = [r["chunk_id"] for r in results]

    # Find rank of correct chunk
    if correct_chunk in retrieved_ids:
        rank = retrieved_ids.index(correct_chunk) + 1
        reciprocal_ranks.append(1 / rank)
    else:
        rank = None
        reciprocal_ranks.append(0)

    # Recall@K
    for k in ks:
        if correct_chunk in retrieved_ids[:k]:
            recall_counts[k] += 1


num_questions = len(candidates)

print("======================================")
print("BASELINE RETRIEVAL EVALUATION")
print("======================================")
print("Questions:", num_questions)

for k in ks:
    recall = recall_counts[k] / num_questions
    print(f"Recall@{k}: {recall:.3f} ({recall*100:.1f}%)")

mrr = np.mean(reciprocal_ranks)

print(f"MRR:       {mrr:.3f}")

BASELINE RETRIEVAL EVALUATION
Questions: 20
Recall@1: 0.600 (60.0%)
Recall@3: 0.800 (80.0%)
Recall@5: 0.900 (90.0%)
Recall@10: 0.900 (90.0%)
MRR:       0.706


In [36]:
from sklearn.metrics import ndcg_score
import numpy as np

def calculate_ndcg(k):
    scores = []

    for qid, results in candidates.items():
        correct_chunk = ground_truth[qid]

        # 1 if relevant, 0 otherwise
        relevance = np.array([
            1 if r["chunk_id"] == correct_chunk else 0
            for r in results[:k]
        ]).reshape(1, -1)

        # Ideal ranking: relevant chunk first
        ideal = np.sort(relevance, axis=1)[:, ::-1]

        score = ndcg_score(relevance, ideal)
        scores.append(score)

    return np.mean(scores)


ndcg5 = calculate_ndcg(5)
ndcg10 = calculate_ndcg(10)

print("======================================")
print("NDCG RESULTS")
print("======================================")
print(f"NDCG@5:  {ndcg5:.3f}")
print(f"NDCG@10: {ndcg10:.3f}")

NDCG RESULTS
NDCG@5:  0.746
NDCG@10: 0.746


In [37]:
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response["message"]["content"])

Hello!


In [38]:
question = "What should you do when driving at sunset or dawn on a dark day?"

# 1. Retrieve relevant handbook chunks
res = col.query(
    query_texts=[question],
    n_results=5
)

context = "\n\n".join(
    f"[Page {meta['page']}]\n{doc}"
    for doc, meta in zip(
        res["documents"][0],
        res["metadatas"][0]
    )
)

# 2. Give the retrieved handbook text to the LLM
prompt = f"""
You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook
information provided below.

Do not use outside knowledge.
Do not invent road rules.

If the answer cannot be found in the provided context,
say: "I cannot find this information in the provided
NSW Road User Handbook."

Include the relevant handbook page in your answer.

HANDBOOK CONTEXT:
{context}

QUESTION:
{question}
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

According to Page 185 of the NSW Road User Handbook, your headlights must be on when driving between sunset and sunrise.


In [39]:
question = "What is the cost of comprehensive car insurance in NSW?"

# 1. Retrieve relevant handbook chunks
res = col.query(
    query_texts=[question],
    n_results=5
)

context = "\n\n".join(
    f"[Page {meta['page']}]\n{doc}"
    for doc, meta in zip(
        res["documents"][0],
        res["metadatas"][0]
    )
)

# 2. Give the retrieved handbook text to the LLM
prompt = f"""
You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook
information provided below.

Do not use outside knowledge.
Do not invent road rules.

If the answer cannot be found in the provided context,
say: "I cannot find this information in the provided
NSW Road User Handbook."

Include the relevant handbook page in your answer.

HANDBOOK CONTEXT:
{context}

QUESTION:
{question}
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

I cannot find this information in the provided NSW Road User Handbook.


In [40]:
out_of_kb = [q for q in golden if q["id"].startswith("o")]

for q in out_of_kb:
    res = col.query(
        query_texts=[q["question"]],
        n_results=5
    )

    context = "\n\n".join(
        f"[Page {meta['page']}]\n{doc}"
        for doc, meta in zip(
            res["documents"][0],
            res["metadatas"][0]
        )
    )

    prompt = f"""
You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook
information provided below.

Do not use outside knowledge.
Do not invent road rules, prices, penalties, or procedures.

If the answer cannot be found in the provided context,
say exactly:

"I cannot find this information in the provided NSW Road User Handbook."

HANDBOOK CONTEXT:
{context}

QUESTION:
{q["question"]}
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    print("=" * 80)
    print(q["id"], "|", q["category"])
    print("QUESTION:", q["question"])
    print("ANSWER:", answer)

o01 | other_jurisdiction
QUESTION: What is the speed limit in a school zone in Victoria?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o02 | unrelated_licence
QUESTION: How do I get a boat licence in NSW?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o03 | other_jurisdiction
QUESTION: What are the demerit point rules in Queensland?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o04 | vehicle_maintenance
QUESTION: How often should I change my engine oil?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o05 | other_jurisdiction
QUESTION: What is the fine for speeding in New Zealand?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o06 | unrelated_licence
QUESTION: How do I apply for a heavy vehicle licence?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
o07 | product_advice
QUESTION: What tyre brand i

In [41]:
inferred = [q for q in golden if q["id"].startswith("i")]

for q in inferred:
    res = col.query(
        query_texts=[q["question"]],
        n_results=5
    )

    context = "\n\n".join(
        f"[Page {meta['page']}]\n{doc}"
        for doc, meta in zip(
            res["documents"][0],
            res["metadatas"][0]
        )
    )

    prompt = f"""
You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook
information provided below.

Do not use outside knowledge.
Do not invent road rules.

If the answer cannot be found in the provided context,
say exactly:

"I cannot find this information in the provided NSW Road User Handbook."

HANDBOOK CONTEXT:
{context}

QUESTION:
{q["question"]}
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    print("=" * 80)
    print(q["id"], "|", q["category"])
    print("QUESTION:", q["question"])
    print("ANSWER:", answer)

i01 | Alcohol and Drugs
QUESTION: Can I drive if I've had a drink while I'm on medication?
ANSWER: I cannot find this information in the provided NSW Road User Handbook.
i02 | General Knowledge
QUESTION: Do I need my headlights on at dusk?
ANSWER: According to page 185 of the NSW Road User Handbook, your headlights must be on:

"...when driving between sunset and sunrise..."

Since the current time is not specified, it is not possible to determine if dusk has occurred.
i03 | General Knowledge
QUESTION: What should I check before getting on the motorway?
ANSWER: According to Page 136 of the NSW Road User Handbook, before driving on a freeway (motorway), you should:

* Make sure your vehicle has enough fuel
* Make sure your vehicle has enough oil
* Make sure your vehicle has enough water
* Check the correct tyre pressure
i04 | General Knowledge
QUESTION: Can I just double park for a minute to grab something?
ANSWER: No, you must not double park.
i05 | Fatigue and Defensive Driving
QUESTI

In [42]:
import json

inferred_results = []

for q in inferred:
    res = col.query(
        query_texts=[q["question"]],
        n_results=5
    )

    context = "\n\n".join(
        f"[Page {meta['page']}]\n{doc}"
        for doc, meta in zip(
            res["documents"][0],
            res["metadatas"][0]
        )
    )

    prompt = f"""
You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook
information provided below.

Do not use outside knowledge.
Do not invent road rules.

If the answer cannot be found in the provided context,
say exactly:

"I cannot find this information in the provided NSW Road User Handbook."

HANDBOOK CONTEXT:
{context}

QUESTION:
{q["question"]}
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    inferred_results.append({
        "id": q["id"],
        "question": q["question"],
        "answer": response["message"]["content"]
    })

with open(
    "data/golden/inferred_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(inferred_results, f, indent=2, ensure_ascii=False)

print("Saved", len(inferred_results), "inferred results")

Saved 10 inferred results
